# Inference and Fitting with a Bayesian Network

Companion notebook for **Causal Networks: Fitting** (6 November 2026).

Five steps, in this order: load, query, sample, fit the parameters, fit the
structure. Every number in the slides comes from this notebook.

The version is pinned because `pgmpy` moves between releases: `get_example_model`,
`map_query` and `ApproxInference.query` have all changed signature in the last two
years. `pydot` is what `nx.nx_pydot.graphviz_layout` needs to draw the graph.

In [ ]:
!pip install "pgmpy==1.1.2" pydot --quiet  # It will take some time ...

## 1. Loading the Alarm network

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from pgmpy.utils import get_example_model

# Load the model (hint: get_example_model).
model = get_example_model("alarm")

print(model)

### The graph

In [ ]:
# Visualization using Graphviz layout (hint: nx.nx_pydot.graphviz_layout).
plt.figure(figsize=(16, 12))
G = nx.DiGraph(model.edges())
nx.draw(
    G,
    pos=nx.nx_pydot.graphviz_layout(G, prog="dot"),
    with_labels=True,
    node_color='lightblue',
    edge_color='black',
    node_size=2000,
    font_size=10,
    font_weight='bold'
)
plt.title("Bayesian Network - NetworkX")
plt.show()

## 2. Structural queries

The graph answers structural questions before any probability. The Markov blanket of
`INTUBATION` is its parents, its children, and the other parents of its children.
Compare the list with the figure on the slides.

In [ ]:
# Parents, children and Markov blanket (hint: get_parents, get_children, get_markov_blanket).
print("parents: ", model.get_parents("INTUBATION"))
print("children:", model.get_children("INTUBATION"))
print("blanket: ", sorted(model.get_markov_blanket("INTUBATION")))

## 3. d-separation

Same graph, two answers. Conditioning on the common cause closes the fork;
conditioning on the common effect opens the collider.

In [ ]:
# d-connection between two nodes, with and without evidence
# (hint: is_dconnected).
print(model.is_dconnected("CVP", "PCWP"))
print(model.is_dconnected("CVP", "PCWP", observed=["LVEDVOLUME"]))

print(model.is_dconnected("HYPOVOLEMIA", "LVFAILURE"))
print(model.is_dconnected("HYPOVOLEMIA", "LVFAILURE", observed=["LVEDVOLUME"]))

## 4. Exact inference

The chain from the slides, with the same numbers: `X -> Y -> Z`, and the answer is
$P(Z) = (0.610, 0.390)$ once `X` and `Y` are summed out. Build it and check that
`pgmpy` agrees with the arithmetic done by hand.

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# The three-node chain of the worked example.
chain = DiscreteBayesianNetwork([("X", "Y"), ("Y", "Z")])
chain.add_cpds(
    TabularCPD("X", 2, [[0.6], [0.4]]),
    TabularCPD("Y", 2, [[0.9, 0.2], [0.1, 0.8]], evidence=["X"], evidence_card=[2]),
    TabularCPD("Z", 2, [[0.8, 0.3], [0.2, 0.7]], evidence=["Y"], evidence_card=[2]),
)
assert chain.check_model()

print(VariableElimination(chain).query(["Z"]))

Now the same call on the network the session is about.

In [ ]:
from pgmpy.inference import VariableElimination

# Construct the inference engine.
inference_engine = VariableElimination(model)

### Compute the marginal probability of P(VENTLUNG)

In [ ]:
# Compute the probability of lung ventilation.
p_ventlung = inference_engine.query(["VENTLUNG"])

print(p_ventlung)

### Compute joint probability of P(VENTLUNG, INTUBATION)

In [ ]:
# Compute the joint probability of lung ventilation and intubation.
p_ventlung_intubation = inference_engine.query(["VENTLUNG", "INTUBATION"])

print(p_ventlung_intubation)

### Compute conditional probability of P(VENTLUNG | INTUBATION = ESOPHAGEAL)

In [ ]:
# Compute the probability of lung ventilation knowing that intubation is esophageal.
p_ventlung_given_intubation = inference_engine.query(
    variables = ["VENTLUNG"],
    evidence = {"INTUBATION": "ESOPHAGEAL"}
)

print(p_ventlung_given_intubation)

### Compute P(VENTLUNG, VENTALV | INTUBATION = ESOPHAGEAL, VENTTUBE = LOW)

In [ ]:
p_vv_ie_vl = inference_engine.query(
    variables = ["VENTLUNG", "VENTALV"],
    evidence = {"INTUBATION": "ESOPHAGEAL", "VENTTUBE": "LOW"}
)

print(p_vv_ie_vl)

### Get the state with maximum probability from the previous queries

MAP assigns jointly. It is not the same as taking the mode of each marginal.

In [ ]:
# MAP(VENTLUNG)
print(inference_engine.map_query(["VENTLUNG"]))
# MAP(VENTLUNG, INTUBATION)
print(inference_engine.map_query(["VENTLUNG", "INTUBATION"]))
# MAP(VENTLUNG | INTUBATION = ESOPHAGEAL)
print(inference_engine.map_query(
    variables = ["VENTLUNG"],
    evidence = {"INTUBATION": "ESOPHAGEAL"}
))
# MAP(VENTLUNG, VENTALV | INTUBATION = ESOPHAGEAL, VENTTUBE = LOW)
print(inference_engine.map_query(
    variables = ["VENTLUNG", "VENTALV"],
    evidence = {"INTUBATION": "ESOPHAGEAL", "VENTTUBE": "LOW"}
))

## 5. Forward sampling

Sampling walks the graph in an order that respects the arrows, so each row is a draw
from the joint. This is where the two halves of the session meet: sampling turns a
network into a table, and fitting turns a table back into a network.

The `seed` is not decoration, it is the only way two people get the same exercise.

In [ ]:
# Draw samples from the joint (hint: simulate).
data = model.simulate(n_samples=1000, seed=42)

print(data.shape)
data.head()

## 6. Approximate inference

`ApproxInference` implements Monte Carlo: draw samples, count.

In [ ]:
from pgmpy.inference import ApproxInference

# Construct the inference engine.
approx_inference_engine = ApproxInference(model)

### Compute P(VENTLUNG) with different samples

In [ ]:
samples = [100, 1_000, 10_000, 100_000]
approx_p_ventlung = []

for n in samples:
    _p = approx_inference_engine.query(
        ["VENTLUNG"],
        n_samples=n,
        seed=42
    )
    print(_p)
    approx_p_ventlung.append(_p)

### Compare with the exact answer

Two traps here. The approximate factor comes back in the order the sampler saw the
states, the exact one in the order the CPD stores them, so the two tables are only
comparable after reindexing on the model's states. And the answer gets better at
$1/\sqrt{n}$, which is slower than it looks.

In [ ]:
# Reindex every approximate answer on the model's state order before comparing.
states = model.states["VENTLUNG"]

all_p_ventlung = pd.DataFrame({
    "APPROX_" + str(n): pd.Series(p.values, index=p.state_names["VENTLUNG"]).reindex(states).values
    for (n, p) in zip(samples, approx_p_ventlung)
})
all_p_ventlung["EXACT"] = p_ventlung.values
all_p_ventlung.index = states

all_p_ventlung

In [ ]:
# Error against the number of samples, exact inference as the reference.
errors = (all_p_ventlung[[c for c in all_p_ventlung.columns if c.startswith("APPROX")]]
          .sub(all_p_ventlung["EXACT"], axis=0).abs().max(axis=0))

ns = [int(c.split("_")[1]) for c in errors.index]

plt.figure(figsize=(7, 4))
plt.plot(ns, errors.values, marker="o", label="measured")
plt.plot(ns, [0.5 / n ** 0.5 for n in ns], "--", label="1/sqrt(n)")
plt.legend()
plt.xlabel("samples $n$"); plt.ylabel("max abs. error")
plt.grid(True, which="major", alpha=0.5)
plt.show()

print(errors)

## 7. Fitting the parameters

The maximum likelihood estimate of a CPD is the frequency of the states, counted per
parent configuration. Here we refit `PCWP` from 200 rows and compare it with the CPD
the network already had: the model is the ground truth, the data is all we would have.

In [ ]:
from pgmpy.estimators import MaximumLikelihoodEstimator

# 200 rows, not the 1000 used for the sampling section: small data is the point here.
fit_data = model.simulate(n_samples=200, seed=7)

# Fit one CPD from data (hint: MaximumLikelihoodEstimator, estimate_cpd).
mle = MaximumLikelihoodEstimator(model, fit_data)
cpd_mle = mle.estimate_cpd("PCWP")

print("true:\n", model.get_cpds("PCWP").get_values().round(3))
print("MLE:\n", cpd_mle.get_values().round(3))
print("exact zeros:", int((cpd_mle.get_values() == 0).sum()))

Three of the nine cells are exactly zero, and one is a saturated 1.00. A zero in a
CPD is an assertion that the event is impossible, and it is the prior's job to prevent
it.

In [ ]:
from pgmpy.estimators import BayesianEstimator

# Fit the same CPD with a Dirichlet prior (hint: BayesianEstimator, prior_type).
bayes = BayesianEstimator(model, fit_data)
cpd_bayes = bayes.estimate_cpd("PCWP", prior_type="BDeu", equivalent_sample_size=5)

print("BDeu:\n", cpd_bayes.get_values().round(3))
print("zeros:", int((cpd_bayes.get_values() == 0).sum()))

### Your turn

Cut the data to 20 rows and fit `PCWP` again with the maximum likelihood estimator.
Look at the *shape* of the returned CPD, not only at its values: a parent state that
never appears in the data disappears from the estimate, and the CPD comes back with
fewer rows than the model needs.

In [ ]:
# MLE on 20 rows: count the rows of the estimate, not only its values.
fit_data_20 = model.simulate(n_samples=20, seed=7)
cpd_20 = MaximumLikelihoodEstimator(model, fit_data_20).estimate_cpd("PCWP")

print("shape:", cpd_20.get_values().shape, "(the model has", model.get_cpds("PCWP").get_values().shape, ")")
cpd_20.get_values().round(3)

## 8. Fitting the structure

Eight nodes and eight edges: small enough that the search finishes in under a second.
`bic-d` is BIC for discrete data, and the result is a DAG, not a model, so it has no
CPDs until you fit them.

The score to report is the **structural Hamming distance**: edges that have to be
added, removed or reversed to turn one graph into the other.

In [ ]:
from pgmpy.estimators import HillClimbSearch
from pgmpy.utils import get_example_model

asia = get_example_model("asia")
asia_data = asia.simulate(n_samples=1000, seed=42)

# Learn the structure (hint: HillClimbSearch, estimate).
learned = HillClimbSearch(asia_data).estimate(scoring_method="bic-d", show_progress=False)

def shd(true_model, learned_dag):
    """Structural Hamming distance between two graphs."""
    return len(set(true_model.edges()) ^ set(learned_dag.edges()))

print("learned edges:", sorted(learned.edges()))
print("SHD:", shd(asia, learned))

### Expert knowledge

A temporal order is a partial order on the variables, and it removes every backward
edge from the search. That is what a domain expert gives you for free.

We use the true order here, which is cheating. The point is how much of the problem
the constraint solves, not where the order came from.

In [ ]:
from pgmpy.causal_discovery import ExpertKnowledge

# A temporal order is a list of tiers, one variable per tier here.
order = [[v] for v in nx.topological_sort(asia)]

ordered = HillClimbSearch(asia_data).estimate(
    scoring_method="bic-d",
    expert_knowledge=ExpertKnowledge(temporal_order=order),
    show_progress=False,
)

print("SHD with the order:", shd(asia, ordered))

### Your turn

Run the unconstrained search with another seed. How much does the graph move? Then
pass the reverse order: the SHD gets worse, and the reason is not that the method is
bad, it is that the constraint is wrong.

In [ ]:
# Another seed, and the reverse order.
other = HillClimbSearch(asia.simulate(n_samples=1000, seed=0)).estimate(
    scoring_method="bic-d", show_progress=False)
print("SHD, seed 0:", shd(asia, other))

reversed_order = [[v] for v in reversed(list(nx.topological_sort(asia)))]
wrong = HillClimbSearch(asia_data).estimate(
    scoring_method="bic-d",
    expert_knowledge=ExpertKnowledge(temporal_order=reversed_order),
    show_progress=False,
)
print("SHD, reversed order:", shd(asia, wrong))

## Takeaways

- Querying is exact when the treewidth allows it, and counting when it does not.
- Fitting parameters is counting with a normalisation. Fitting them badly is also
  counting, and it produces zeros where the data said nothing.
- A prior says how much a missing observation is worth; the equivalent sample size is
  how you quantify it.
- Fitting the structure is a search over graphs, scored by the same likelihood.
  Constraints from the domain are cheaper than data.
- `causal_discovery_with_cdt.ipynb` runs the same task with the Causal Discovery
  Toolbox. It needs an R runtime, so it is homework, not this slot.